# Capstone Project Asah 2026 - Tim A26-CS011
## Sistem Manajemen Pesanan, Keuangan, dan Analitik pada Tongkrongan Hiroo
### Modul AI Engineering & Data Science: Peramalan Penjualan Menu (Time Series Forecasting)

---
### 📌 Ringkasan Masalah & Konteks Bisnis
**Tongkrongan Hiroo** adalah kafe di bidang F&B dengan menu makanan, minuman, dan snack berharga Rp5.000 – Rp25.000. 
Berdasarkan wawancara dengan pemilik (Kak Faiz):
1. Pencatatan transaksi dan stok masih dilakukan secara manual menggunakan buku tulis.
2. Saat jam ramai (*peak hours*), terutama **malam minggu saat ada live music**, pengunjung membeludak dan kasir keteteran. Alur pemesanan menjadi acak dan banyak transaksi terlewat dicatat.
3. Timbul selisih uang harian mencapai sekitar **Rp600.000 dalam satu malam** serta kesulitan memantau stok dan tren penjualan.

### 🎯 Tujuan Pengembangan Model
Membangun pipeline Machine Learning terstruktur untuk **memprediksi volume penjualan porsi per menu per hari**:
1. Menyuplai estimasi penjualan porsi ke **Dashboard Owner** untuk perencanaan stok dan pemantauan bisnis.
2. Menyuplai data terstruktur (menu populer vs menu lambat laku) ke modul **Generative AI** untuk menyusun narasi strategi bisnis dan rekomendasi promo *bundling*.
3. Melakukan **Tri-Model Benchmark** komparatif antara:
   - **Model 1**: *Facebook Prophet* (Multivariate Time Series dengan Regressor)
   - **Model 2**: *Random Forest Regressor* (Bagging Ensemble)
   - **Model 3**: *XGBoost Regressor* (Gradient Boosting / NARX)
4. Menerapkan metodologi data science yang ketat: **Time-Based Split (mencegah Data Leakage)**, eliminasi fitur redundan, metrik evaluasi komprehensif (MAE, RMSE, $R^2$, MAPE), dan *model serialization* untuk diintegrasikan ke backend FastAPI.


## 1. Setup Lingkungan & Pemuatan Pustaka (Library Imports)

Pada tahap ini, kita memuat seluruh pustaka yang diperlukan:
- **Pengolahan Data**: `pandas`, `numpy`
- **Visualisasi**: `matplotlib`, `seaborn`
- **Machine Learning & Evaluasi**: `scikit-learn` (One-Hot Encoder, Random Forest, Metrics)
- **Gradient Boosting**: `xgboost`
- **Time Series Khusus**: `prophet`
- **Penyimpanan Model**: `joblib`
- **Download Data dari GDrive**: `gdown`


In [ ]:
# Instalasi dependensi tambahan jika dijalankan di Google Colab
# !pip install -q gdown xgboost prophet scikit-learn

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import gdown

# Modeling & Metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor
from prophet import Prophet

# Konfigurasi visualisasi
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(" Seluruh pustaka berhasil dimuat dengan sukses!")


## 2. Pemuatan Dataset Langsung dari Google Drive

Sesuai instruksi proyek, dataset dummy transaksi kafe dimuat secara dinamis langsung dari tautan **Google Drive** menggunakan File ID `1VU59lK9y-scT_yvLhyzVlg6HdFy4aDDA`.

Metode ini memastikan notebook dapat dijalankan secara portabel baik di Google Colab maupun di komputer lokal tanpa perlu mengunggah file CSV secara manual.


In [ ]:
file_id = '1VU59lK9y-scT_yvLhyzVlg6HdFy4aDDA'
gdrive_url = f'https://drive.google.com/uc?id={file_id}'
csv_filename = 'dataset_dummy_tongkrongan_hiroo_v3.csv'

# Unduh dataset jika belum tersedia secara lokal di working directory
if not os.path.exists(csv_filename):
    print(f"Mengunduh {csv_filename} dari Google Drive...")
    try:
        gdown.download(gdrive_url, csv_filename, quiet=False)
    except Exception as e:
        print(f"Gdown fallback ke direct download: {e}")
        direct_url = f'https://drive.google.com/uc?export=download&id={file_id}'
        df_raw = pd.read_csv(direct_url)
        df_raw.to_csv(csv_filename, index=False)

# Baca dataset ke dalam DataFrame pandas
df_raw = pd.read_csv(csv_filename)

print(f"\n Dataset berhasil dimuat!")
print(f"Dimensi Data: {df_raw.shape[0]} baris transaksi, {df_raw.shape[1]} kolom.")
df_raw.head()


## 3. Pemeriksaan Struktur Data & Pembersihan Awal (Data Inspection & Cleaning)

Memeriksa tipe data, kelengkapan nilai (*missing values*), data duplikat, dan standarisasi nama kolom.


In [ ]:
# Standarisasi nama kolom (huruf kecil dan ganti spasi dengan underscore)
df_raw.columns = df_raw.columns.str.strip().str.lower().str.replace(' ', '_')
print("Kolom yang tersedia dalam dataset mentah:")
print(list(df_raw.columns))

# Pemeriksaan info umum dan missing values
print("\n--- Ringkasan Tipe Data & Missing Values ---")
print(df_raw.info())
print("\nJumlah nilai kosong per kolom:")
print(df_raw.isnull().sum())

# Pemeriksaan duplikasi data
duplicate_count = df_raw.duplicated().sum()
print(f"\nJumlah baris duplikat identik: {duplicate_count}")
if duplicate_count > 0:
    df_raw = df_raw.drop_duplicates()
    print("Baris duplikat berhasil dihapus.")

# Deteksi nama kolom tanggal / waktu
date_col_candidates = [col for col in df_raw.columns if any(kw in col for kw in ['tanggal', 'date', 'waktu', 'time'])]
date_col = date_col_candidates[0] if date_col_candidates else df_raw.columns[0]
print(f"\nKolom tanggal terdeteksi: '{date_col}'")

# Konversi kolom tanggal langsung ke format datetime (dinormalisasi ke 00:00:00 tanpa jam/menit)
df_raw['tanggal'] = pd.to_datetime(df_raw[date_col]).dt.normalize()

print(f"Rentang tanggal transaksi: {df_raw['tanggal'].min().strftime('%d %b %Y')} s.d. {df_raw['tanggal'].max().strftime('%d %b %Y')}")
print(f"Total rentang waktu: {(df_raw['tanggal'].max() - df_raw['tanggal'].min()).days + 1} hari kalender.")


## 4. Analisis Eksplorasi Data (Exploratory Data Analysis - EDA)

Pada tahap ini, kita menganalisis karakteristik data secara mendalam melalui 4 visualisasi terpisah:
1. **Peringkat Menu**: Identifikasi *Top 5 Best-Selling Menus* vs *5 Slowest-Moving Menus*.
2. **Tren Penjualan Harian**: Memantau volume penjualan porsi kafe dari waktu ke waktu.
3. **Pola Mingguan (Weekly Seasonality)**: Memeriksa lonjakan pengunjung pada hari Sabtu malam (*Live Music*).
4. **Proporsi Pembayaran**: Mengetahui rasio transaksi Tunai vs QRIS.


In [ ]:
# Identifikasi kolom nama menu dan kuantitas porsi terjual
menu_col = [col for col in df_raw.columns if any(kw in col for kw in ['menu', 'item', 'produk'])][0]
qty_col = [col for col in df_raw.columns if any(kw in col for kw in ['qty', 'jumlah', 'kuantitas', 'porsi'])][0]

print(f"Kolom identitas menu terdeteksi : '{menu_col}'")
print(f"Kolom kuantitas porsi terdeteksi: '{qty_col}'")


### 4.1 Visualisasi 1: Top 5 Menu Terlaris vs 5 Menu Lambat Laku

Visualisasi ini sangat penting bagi pemilik usaha (Kak Faiz) dan menjadi input faktual bagi modul **Generative AI** untuk merumuskan strategi promo diskon *bundling* (memasangkan menu lambat laku dengan menu populer).


In [ ]:
# Agregasi total penjualan per menu
menu_sales = df_raw.groupby(menu_col)[qty_col].sum().sort_values(ascending=False)
top_bottom_menu = pd.concat([menu_sales.head(5), menu_sales.tail(5)])
palette_rank = ['#2b5c8f'] * 5 + ['#d95f02'] * 5

plt.figure(figsize=(12, 6))
sns.barplot(x=top_bottom_menu.values, y=top_bottom_menu.index, palette=palette_rank)
plt.title("Top 5 Menu Terlaris (Biru) vs 5 Menu Lambat Laku (Oranye) pada Tongkrongan Hiroo", fontsize=13, fontweight='bold')
plt.xlabel("Total Porsi Terjual (Seluruh Periode)")
plt.ylabel("Nama Menu")
for i, v in enumerate(top_bottom_menu.values):
    plt.text(v + (max(top_bottom_menu.values) * 0.01), i, f"{v:,} porsi", va='center', fontsize=10, fontweight='semibold')

plt.tight_layout()
plt.show()

print("Top 5 Menu Terlaris:")
for m, q in menu_sales.head(5).items():
    print(f" - {m}: {q} porsi")
print("\n5 Menu Paling Lambat Laku:")
for m, q in menu_sales.tail(5).items():
    print(f" - {m}: {q} porsi")


### 4.2 Visualisasi 2: Tren Total Penjualan Porsi Harian Kafe

Memantau fluktuasi total penjualan kafe dari hari ke hari untuk mendeteksi tren umum, kestabilan bisnis, dan lonjakan siklikal.


In [ ]:
# Agregasi total penjualan per hari kalender
daily_overall = df_raw.groupby('tanggal')[qty_col].sum()

plt.figure(figsize=(14, 5))
plt.plot(daily_overall.index, daily_overall.values, color='#1b9e77', linewidth=2, marker='o', markersize=4, label='Total Porsi Harian')
plt.axhline(daily_overall.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Rata-rata: {daily_overall.mean():.1f} porsi/hari')

plt.title("Tren Pergerakan Total Penjualan Porsi Harian Tongkrongan Hiroo", fontsize=13, fontweight='bold')
plt.xlabel("Tanggal")
plt.ylabel("Total Porsi Terjual")
plt.xticks(rotation=30)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Rata-rata penjualan harian: {daily_overall.mean():.1f} porsi")
print(f"Penjualan harian tertinggi : {daily_overall.max()} porsi (pada {daily_overall.idxmax().strftime('%d %B %Y')})")
print(f"Penjualan harian terendah  : {daily_overall.min()} porsi (pada {daily_overall.idxmin().strftime('%d %B %Y')})")


### 4.3 Visualisasi 3: Pola Penjualan Berdasarkan Hari dalam Seminggu (Weekly Seasonality)

Membuktikan secara empiris perbedaan volume pesanan antara hari kerja (*weekday*) dengan akhir pekan, khususnya memverifikasi keterangan pemilik mengenai lonjakan drastis pada hari **Sabtu malam saat acara Live Music**.


In [ ]:
# Pemetaan nama hari ke bahasa Indonesia
df_raw['hari_nama'] = df_raw['tanggal'].dt.day_name()
day_order_en = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hari_map_id = {
    'Monday': 'Senin', 'Tuesday': 'Selasa', 'Wednesday': 'Rabu',
    'Thursday': 'Kamis', 'Friday': 'Jumat', 'Saturday': 'Sabtu (Live Music)', 'Sunday': 'Minggu'
}
df_raw['hari_id'] = df_raw['hari_nama'].map(hari_map_id)
day_order_id = [hari_map_id[d] for d in day_order_en]

# Hitung rata-rata penjualan per hari
day_sales = df_raw.groupby('hari_id')[qty_col].mean().reindex(day_order_id)
palette_days = ['#74add1'] * 5 + ['#d73027', '#fdae61']  # Sorot Sabtu dengan warna merah

plt.figure(figsize=(12, 5))
sns.barplot(x=day_sales.index, y=day_sales.values, palette=palette_days)
plt.title("Rata-Rata Penjualan Porsi per Transaksi Berdasarkan Hari (Sabtu Live Music Melonjak)", fontsize=13, fontweight='bold')
plt.xlabel("Hari dalam Seminggu")
plt.ylabel("Rata-rata Porsi per Transaksi")
for i, v in enumerate(day_sales.values):
    plt.text(i, v + 0.05, f"{v:.2f}", ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("Kesimpulan Pola Mingguan:")
print("- Hari Senin sampai Kamis memiliki pola penjualan yang relatif stabil dan tenang.")
print("- Hari Sabtu (Live Music) menunjukkan lonjakan tertinggi, menjadi pemicu utama staf kasir kewalahan jika pencatatan masih manual.")


### 4.4 Visualisasi 4: Distribusi Metode Pembayaran (Tunai vs QRIS)

Melihat preferensi metode transaksi pelanggan di Tongkrongan Hiroo untuk membantu pemilik kafe dalam rekonsiliasi kas fisik vs mutasi rekening digital.


In [ ]:
payment_col_candidates = [c for c in df_raw.columns if any(kw in c for kw in ['bayar', 'metode', 'payment'])]

if payment_col_candidates:
    pay_col = payment_col_candidates[0]
    pay_counts = df_raw[pay_col].value_counts()

    plt.figure(figsize=(7, 7))
    plt.pie(
        pay_counts.values,
        labels=pay_counts.index,
        autopct='%1.1f%%',
        colors=['#2b5c8f', '#41b6c4', '#fdae61'],
        startangle=140,
        explode=[0.05] * len(pay_counts),
        shadow=True,
        textprops={'fontsize': 11, 'fontweight': 'bold'}
    )
    plt.title(f"Proporsi Penggunaan Metode Pembayaran ({pay_col})", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"Rincian Transaksi per Metode Pembayaran:")
    for m, c in pay_counts.items():
        print(f" - {m}: {c:,} transaksi ({c / len(df_raw) * 100:.1f}%)")
else:
    print("Kolom metode pembayaran tidak ditemukan dalam dataset mentah.")


## 5. Agregasi Harian per Menu & Pembentukan Deret Waktu

Karena model bertujuan meramal **total porsi terjual per menu per hari**, data transaksi kasir mentah diagregasi ke level `(tanggal, nama_menu)`.

Kita juga melakukan *grid completion* untuk memastikan setiap tanggal memiliki data untuk seluruh menu (jika menu tidak laku pada tanggal tersebut, porsinya diisi `0`).


In [ ]:
# Agregasi total kuantitas porsi terjual per tanggal dan per menu
df_daily = df_raw.groupby(['tanggal', menu_col])[qty_col].sum().reset_index()
df_daily.rename(columns={menu_col: 'nama_menu', qty_col: 'total_qty'}, inplace=True)

# Grid Completion: Membuat kombinasi penuh setiap tanggal x setiap menu
all_dates = pd.date_range(start=df_daily['tanggal'].min(), end=df_daily['tanggal'].max(), freq='D')
all_menus = df_daily['nama_menu'].unique()

full_index = pd.MultiIndex.from_product([all_dates, all_menus], names=['tanggal', 'nama_menu'])
df_daily = df_daily.set_index(['tanggal', 'nama_menu']).reindex(full_index, fill_value=0).reset_index()

print(f"Total baris data deret waktu harian: {len(df_daily)} baris ({len(all_dates)} hari x {len(all_menus)} menu).")
df_daily.head(10)


## 6. Rekayasa Fitur & Seleksi Fitur (Feature Engineering & Selection)

Sesuai rencana implementasi yang telah dianalisis secara kritis:
1. **Fitur Kalender yang Dipertahankan**:
   - `day_of_week` (0=Senin, ..., 6=Minggu)
   - `is_live_music_night` (1 jika hari Sabtu malam saat event Hiroo, 0 lainnya)
2. **Fitur yang DIBUANG (*Eliminated*)**:
   - `is_weekend` (100% redundan dengan `day_of_week >= 5`)
   - `kategori` (100% redundan karena sudah diwakili oleh One-Hot `nama_menu`)
   - `harga_satuan` (statis per menu, redundan dengan identitas menu)
   - `metode_pembayaran` & `subtotal` (kebocoran target / *target leakage*)
3. **Fitur Deret Waktu (*Time Series Signals*)**:
   - `qty_lag_7`: Penjualan menu pada hari yang sama di minggu sebelumnya.
   - `rolling_mean_7`: Rata-rata bergerak 7 hari terakhir (menggunakan `.shift(1)` agar **tidak mengintip hari ini / zero leakage**).
4. **Encoding Menu**:
   - *One-Hot Encoding* pada `nama_menu` untuk menjaga transparansi dan independensi perilaku tiap menu tanpa target leakage.


In [ ]:
# Urutkan berdasarkan nama menu dan tanggal sebelum kalkulasi lag
df_daily = df_daily.sort_values(by=['nama_menu', 'tanggal']).reset_index(drop=True)

# 1. Fitur Kalender & Event Hiroo
df_daily['day_of_week'] = df_daily['tanggal'].dt.dayofweek
df_daily['is_live_music_night'] = (df_daily['day_of_week'] == 5).astype(int)

# 2. Fitur Deret Waktu: Lag 7 dan Rolling Mean 7
# shift(1) memastikan rolling_mean hanya menggunakan riwayat masa lalu (mencegah leakage!)
df_daily['qty_lag_7'] = df_daily.groupby('nama_menu')['total_qty'].shift(7)
df_daily['rolling_mean_7'] = df_daily.groupby('nama_menu')['total_qty'].transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=1).mean()
)

# Buang 7 hari pertama yang menghasilkan NaN akibat lag_7 (warmup period)
df_model = df_daily.dropna().copy().reset_index(drop=True)

# 3. One-Hot Encoding pada nama_menu
df_encoded = pd.get_dummies(df_model, columns=['nama_menu'], prefix='menu', dtype=int)

print(f"Dimensi data siap latih setelah lag & encoding: {df_encoded.shape}")
print(f"Daftar kolom fitur final:")
for c in df_encoded.columns:
    if c not in ['tanggal', 'total_qty']:
        print(f" - {c}")

df_encoded.head()


## 7. Pemisahan Data Berbasis Waktu (Time-Based Train-Test Split)

> **Mencegah Data Leakage**: Pada peramalan deret waktu, pembagian data **TIDAK BOLEH DIACOK** (`shuffle=False`). 
> Kita menggunakan pemotongan kronologis: **80% periode awal sebagai Data Latih (Train)** dan **20% periode akhir sebagai Data Uji (Test)**. 
> Ini mensimulasikan kondisi nyata pemilik kafe yang meramal penjualan hari esok murni bermodalkan riwayat masa lalu.


In [ ]:
# Tentukan pemotongan tanggal kronologis (80% Train, 20% Test)
unique_dates = np.sort(df_encoded['tanggal'].unique())
split_idx = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates[split_idx]

train_df = df_encoded[df_encoded['tanggal'] < cutoff_date].copy()
test_df = df_encoded[df_encoded['tanggal'] >= cutoff_date].copy()

# Pisahkan variabel prediktor (X) dan target (y)
feature_cols = [c for c in df_encoded.columns if c not in ['tanggal', 'total_qty']]

X_train, y_train = train_df[feature_cols], train_df['total_qty']
X_test, y_test = test_df[feature_cols], test_df['total_qty']

print(f" Tanggal Cutoff Pembagian Data: {pd.to_datetime(cutoff_date).strftime('%d %B %Y')}")
print(f"Periode Training : {train_df['tanggal'].min().strftime('%d %b %Y')} s.d. {train_df['tanggal'].max().strftime('%d %b %Y')} ({len(train_df)} baris)")
print(f"Periode Testing  : {test_df['tanggal'].min().strftime('%d %b %Y')} s.d. {test_df['tanggal'].max().strftime('%d %b %Y')} ({len(test_df)} baris)")
print(f"Jumlah Fitur Prediktor (X): {len(feature_cols)} fitur.")


## 8. Pelatihan Model (Tri-Model Benchmark)

Kita melatih dan menguji 3 paradigma pemodelan yang berbeda:
1. **Model 1: Facebook Prophet (Multivariate Time Series)**
   - Model aditif khusus deret waktu dengan regressor eksternal `is_live_music_night` dan `weekly_seasonality=True`.
2. **Model 2: Random Forest Regressor (`scikit-learn`)**
   - Paradigma *Bagging Ensemble* berbasis pohon keputusan paralel.
3. **Model 3: XGBoost Regressor (`xgboost`)**
   - Paradigma *Gradient Boosting (NARX)* yang mengoreksi galat secara iteratif sekuensial.


In [ ]:
# ==============================================================================
# Model 1: Facebook Prophet (Multivariate Time Series Benchmark)
# ==============================================================================
print("1. Melatih Model 1: Facebook Prophet (Multivariate)...")

# Untuk perbandingan yang adil, kita uji Prophet pada agregat penjualan menu utama atau per menu
# Menyiapkan data Prophet dengan format kolom 'ds' dan 'y'
prophet_train = df_model[df_model['tanggal'] < cutoff_date][['tanggal', 'total_qty', 'is_live_music_night']].copy()
prophet_train.rename(columns={'tanggal': 'ds', 'total_qty': 'y'}, inplace=True)

prophet_test = df_model[df_model['tanggal'] >= cutoff_date][['tanggal', 'total_qty', 'is_live_music_night']].copy()
prophet_test.rename(columns={'tanggal': 'ds', 'total_qty': 'y'}, inplace=True)

# Inisialisasi dan fit Prophet dengan regressor tambahan
model_prophet = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
model_prophet.add_regressor('is_live_music_night')
model_prophet.fit(prophet_train)

# Prediksi pada rentang data test
future_df = prophet_test[['ds', 'is_live_music_night']].copy()
prophet_forecast = model_prophet.predict(future_df)
y_pred_prophet = np.maximum(0, prophet_forecast['yhat'].values)
print("   Prophet selesai dilatih dan diprediksi.")

# ==============================================================================
# Model 2: Random Forest Regressor (Bagging Ensemble)
# ==============================================================================
print("\n2. Melatih Model 2: Random Forest Regressor (scikit-learn)...")
model_rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model_rf.fit(X_train, y_train)
y_pred_rf = np.maximum(0, model_rf.predict(X_test))
print("   Random Forest selesai dilatih.")

# ==============================================================================
# Model 3: XGBoost Regressor (Gradient Boosting NARX)
# ==============================================================================
print("\n3. Melatih Model 3: XGBoost Regressor (xgboost)...")
model_xgb = XGBRegressor(
    n_estimators=150,
    learning_rate=0.07,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    objective='reg:squarederror'
)
model_xgb.fit(X_train, y_train)
y_pred_xgb = np.maximum(0, model_xgb.predict(X_test))
print("   XGBoost selesai dilatih.")


## 9. Evaluasi & Perbandingan Model (Benchmarking)

Mengevaluasi performa ketiga model pada Data Uji (*Test Set*) menggunakan metrik standar industri:
- **MAE (*Mean Absolute Error*)**: Rata-rata selisih porsi nyata antara prediksi dan penjualan aktual (semakin kecil semakin baik).
- **RMSE (*Root Mean Squared Error*)**: Besaran galat kuadratik untuk mengukur penalti kesalahan besar.
- **$R^2$ (*Coefficient of Determination*)**: Proporsi variansi data yang berhasil ditangkap model (maksimal 1.0).
- **MAPE (*Mean Absolute Percentage Error*)**: Persentase rata-rata deviasi relatif.


In [ ]:
def calculate_metrics(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    # Hindari pembagian nol pada MAPE
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {
        'Model': model_name,
        'MAE (Porsi)': round(mae, 2),
        'RMSE (Porsi)': round(rmse, 2),
        'R2 Score': round(r2, 4),
        'MAPE (%)': round(mape, 2)
    }

results = [
    calculate_metrics(prophet_test['y'].values, y_pred_prophet, 'Facebook Prophet (Multivariate)'),
    calculate_metrics(y_test, y_pred_rf, 'Random Forest Regressor (Bagging)'),
    calculate_metrics(y_test, y_pred_xgb, 'XGBoost Regressor (Gradient Boosting)')
]

df_results = pd.DataFrame(results)
print("=== TABEL HASIL BENCHMARK MODEL PREDIKSI PENJUALAN HIROO ===")
display(df_results)


### Visualisasi Komparasi Hasil Prediksi vs Aktual

Membandingkan kurva prediksi ketiga model terhadap data aktual pada data uji.


In [ ]:
# Gabungkan prediksi ke dataframe test untuk visualisasi agregat harian
test_vis = test_df[['tanggal', 'total_qty']].copy()
test_vis['pred_rf'] = y_pred_rf
test_vis['pred_xgb'] = y_pred_xgb

daily_test_actual = test_vis.groupby('tanggal')['total_qty'].sum()
daily_test_rf = test_vis.groupby('tanggal')['pred_rf'].sum()
daily_test_xgb = test_vis.groupby('tanggal')['pred_xgb'].sum()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 1. Kurva Penjualan Harian Aktual vs Prediksi
axes[0].plot(daily_test_actual.index, daily_test_actual.values, 'k-o', label='Aktual (Penjualan Nyata)', linewidth=2.5)
axes[0].plot(daily_test_rf.index, daily_test_rf.values, 'b--', label='Prediksi Random Forest', linewidth=2)
axes[0].plot(daily_test_xgb.index, daily_test_xgb.values, 'g-.', label='Prediksi XGBoost', linewidth=2)
axes[0].set_title("Komparasi Total Penjualan Porsi Harian: Aktual vs Model Prediksi pada Periode Testing", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Tanggal")
axes[0].set_ylabel("Total Porsi Terjual")
axes[0].legend(fontsize=11)

# 2. Distribusi Residual Error (Aktual - Prediksi)
residuals_rf = y_test - y_pred_rf
residuals_xgb = y_test - y_pred_xgb

sns.kdeplot(residuals_rf, label=f'Residual RF (MAE: {df_results.loc[1, "MAE (Porsi)"]})', color='blue', ax=axes[1], fill=True, alpha=0.2)
sns.kdeplot(residuals_xgb, label=f'Residual XGBoost (MAE: {df_results.loc[2, "MAE (Porsi)"]})', color='green', ax=axes[1], fill=True, alpha=0.2)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title("Distribusi Residual Galat (Error): Semakin Rapat ke Nol Semakin Akurat", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Selisih Porsi (Aktual - Prediksi)")
axes[1].set_ylabel("Densitas")
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()


## 10. Analisis Kepentingan Fitur (Feature Importance)

Menganalisis fitur apa saja yang memiliki pengaruh paling signifikan terhadap volume penjualan menu di Tongkrongan Hiroo berdasarkan model pohon terbaik.


In [ ]:
# Feature Importance dari XGBoost
xgb_importance = pd.Series(model_xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(x=xgb_importance.head(12).values, y=xgb_importance.head(12).index, palette='viridis')
plt.title("Top 12 Fitur Paling Berpengaruh Terhadap Prediksi Penjualan (XGBoost)", fontsize=13, fontweight='bold')
plt.xlabel("Tingkat Kepentingan Fitur (Feature Importance Score)")
plt.ylabel("Nama Fitur")
plt.tight_layout()
plt.show()

print("Interpretasi Bisnis:")
print("1. Riwayat porsi minggu lalu ('qty_lag_7') dan tren moving average ('rolling_mean_7') mendominasi akurasi.")
print("2. 'is_live_music_night' dan 'day_of_week' menjadi pendorong utama lonjakan pesanan akhir pekan di kafe Hiroo.")


## 11. Penyimpanan Model & Simulasi Fungsi Inferensi JSON (FastAPI / Gen AI Serving)

Model terbaik disimpan ke dalam file `.joblib` agar siap dimuat oleh backend **FastAPI**.
Kita juga membuat fungsi simulasi inferensi `predict_sales_forecast(menu_name, days_ahead)` yang menghasilkan payload JSON terstruktur untuk dikonsumsi oleh:
1. **Dashboard Owner Web**: Menampilkan grafik prediksi harian ke depan.
2. **Modul Generative AI**: Menjadi input data faktual untuk merumuskan promo diskon *bundling* menu lambat laku dengan menu populer.


In [ ]:
# Simpan model terbaik dan daftar nama fitur
model_artifact_path = 'best_sales_forecasting_model.joblib'
feature_artifact_path = 'model_feature_columns.joblib'

best_model = model_xgb if df_results.loc[2, 'MAE (Porsi)'] <= df_results.loc[1, 'MAE (Porsi)'] else model_rf
joblib.dump(best_model, model_artifact_path)
joblib.dump(feature_cols, feature_artifact_path)

print(f" Model terbaik berhasil diekspor ke: {model_artifact_path}")
print(f" Daftar fitur prediktor berhasil diekspor ke: {feature_artifact_path}")

# ==============================================================================
# Fungsi Inferensi Prediksi untuk Integrasi Backend / Gen AI
# ==============================================================================
def predict_sales_forecast(menu_name, days_ahead=7):
    """
    Fungsi inferensi untuk memprediksi penjualan masa depan suatu menu.
    Output diformat dalam struktur JSON standar yang siap dikonsumsi Frontend & Gen AI.
    """
    menu_col_name = f'menu_{menu_name}'
    if menu_col_name not in feature_cols:
        return {'status': 'error', 'message': f'Menu "{menu_name}" tidak ditemukan dalam basis data.'}

    last_date = df_daily['tanggal'].max()
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=days_ahead, freq='D')
    
    # Ambil nilai lag_7 dan rolling_mean_7 terbaru dari data riil
    recent_menu_data = df_daily[df_daily['nama_menu'] == menu_name].sort_values('tanggal').tail(7)
    recent_rolling = recent_menu_data['total_qty'].mean()
    recent_lag = recent_menu_data['total_qty'].iloc[-1]

    forecast_records = []
    total_predicted = 0

    for f_date in future_dates:
        dow = f_date.dayofweek
        is_live = 1 if dow == 5 else 0
        
        # Bangun vektor input 1 baris sesuai skema fitur
        input_data = {col: 0 for col in feature_cols}
        input_data['day_of_week'] = dow
        input_data['is_live_music_night'] = is_live
        input_data['qty_lag_7'] = recent_lag
        input_data['rolling_mean_7'] = recent_rolling
        input_data[menu_col_name] = 1

        input_df = pd.DataFrame([input_data])
        pred_qty = int(round(max(0, best_model.predict(input_df)[0])))
        total_predicted += pred_qty

        forecast_records.append({
            'date': f_date.strftime('%Y-%m-%d'),
            'day': f_date.day_name(),
            'predicted_qty': pred_qty
        })

    avg_qty = round(total_predicted / days_ahead, 1)
    status_label = "popular" if avg_qty >= 25 else "slow_moving"

    return {
        'status': 'success',
        'menu_name': menu_name,
        'forecast_period_days': days_ahead,
        'predictions': forecast_records,
        'summary': {
            'total_predicted_qty': total_predicted,
            'avg_daily_predicted_qty': avg_qty,
            'status': status_label
        }
    }

# Simulasi uji fungsi inferensi untuk menu Mie Goreng Hiroo
sample_forecast = predict_sales_forecast('Mie Goreng Hiroo', days_ahead=7)
print("\n=== HASIL SIMULASI INFERENSI JSON UNTUK FASTAPI & GENERATIVE AI ===")
import pprint
pprint.pprint(sample_forecast)


## 12. Kesimpulan & Langkah Integrasi Selanjutnya

###  Ringkasan Capaian Teknis:
1. **Data Loading via GDrive URL**: Berhasil dimuat secara dinamis tanpa ketergantungan file lokal statis.
2. **Pembersihan & Anti-Leakage**: Menghapus `is_weekend`, `kategori`, `harga_satuan`, `subtotal`, dan `metode_pembayaran` untuk menjamin model bebas bias kebocoran data.
3. **Pemisahan Waktu (Time-Based Split)**: Membagi data secara kronologis (80% Train, 20% Test) untuk mensimulasikan kondisi bisnis riil.
4. **Tri-Model Benchmark**: Menghasilkan perbandingan empiris antara *Prophet*, *Random Forest*, dan *XGBoost*.
5. **Model Serialization**: Model terbaik disimpan ke `.joblib` beserta skema fitur prediktor.

### 🔗 Langkah Selanjutnya (Sesuai Project Plan Minggu 2-3):
- **Integrasi Backend (Dwi & Rilo)**: Membuat endpoint REST API menggunakan **FastAPI** (`/api/v1/predict/menu`) yang memuat artefak `best_sales_forecasting_model.joblib`.
- **Integrasi Generative AI (Rilo)**: Menghubungkan output JSON menu populer & menu lambat laku ke modul LLM prompt engineering untuk menghasilkan narasi strategi bisnis dan saran promo *bundling* otomatis pada Dashboard Owner.
